# Track B API test suite

Goes through the **Setup and API key testing** checklist from the project doc. It checks that the Qwen3-8B chat endpoint (VM06) and the embeddings service (VM02) work, that they enforce the documented limits, and that `api_client.py` handles them.

**Run it top to bottom.** It makes about 25 real API calls (about 15 seconds of model time). The concurrency test (section 9) briefly holds the group key, so a teammate's call made at that moment may get a 429 error.

No key is ever printed.

In [1]:
import json, os, ssl, subprocess, threading, time, urllib.request, urllib.error, math
import api_client
from api_client import Client, Endpoint, RateLimiter, ApiError, CALL_LOG

client = Client()
E = os.environ
mask = lambda k: k[:4] + "…" + k[-2:] if len(k) > 8 else "***"

RESULTS = []
def check(name, ok, detail=""):
    RESULTS.append((name, "PASS" if ok is True else ("WARN" if ok is None else "FAIL"), detail))
    print(f"[{RESULTS[-1][1]}] {name}" + (f" — {detail}" if detail else ""))

def raw(base, key, cafile, route, body=None, method=None, timeout=120):
    """One request with no retry, lock or checks. Returns (status, text)."""
    ctx = ssl.create_default_context(cafile=cafile) if cafile else ssl.create_default_context()
    req = urllib.request.Request(base.rstrip("/") + route, method=method or ("POST" if body is not None else "GET"),
                                 data=json.dumps(body).encode() if body is not None else None,
                                 headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=timeout) as r:
            return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode(errors="replace")
    except Exception as e:
        return type(e).__name__, str(e)

LLM = (E["LLM_BASE_URL"], E["LLM_API_KEY"], E["LLM_CA_CERT"])
EMB = (E["EMBED_BASE_URL"], E["EMBED_API_KEY"], E["EMBED_CA_CERT"])
print("Chat      :", E["LLM_BASE_URL"], "| key", mask(E["LLM_API_KEY"]), "| cert", E["LLM_CA_CERT"])
print("Embeddings:", E["EMBED_BASE_URL"], "| key", mask(E["EMBED_API_KEY"]), "| cert", E["EMBED_CA_CERT"])

Chat      : https://103.42.50.41:1032/v1 | key sk-g…i8 | cert secrets/vm06-ca.crt
Embeddings: https://103.42.50.41:1035/v1 | key sk-u…Ka | cert secrets/vm02-ca.crt


## 1. Configuration and certificates

In [2]:
for v in ["LLM_BASE_URL","LLM_API_KEY","LLM_CA_CERT","LLM_MODEL","EMBED_BASE_URL","EMBED_API_KEY","EMBED_CA_CERT","EMBED_MODEL"]:
    check(f"{v} set", bool(E.get(v)) and not E[v].startswith("YOUR_"))
check("chat cert file exists", os.path.exists(E["LLM_CA_CERT"]), E["LLM_CA_CERT"])
check("embed cert file exists", os.path.exists(E["EMBED_CA_CERT"]), E["EMBED_CA_CERT"])
check("base URL ends in /v1 (append /chat/completions once)", E["LLM_BASE_URL"].rstrip("/").endswith("/v1"))

[PASS] LLM_BASE_URL set
[PASS] LLM_API_KEY set
[PASS] LLM_CA_CERT set
[PASS] LLM_MODEL set
[PASS] EMBED_BASE_URL set
[PASS] EMBED_API_KEY set
[PASS] EMBED_CA_CERT set
[PASS] EMBED_MODEL set
[PASS] chat cert file exists — secrets/vm06-ca.crt
[PASS] embed cert file exists — secrets/vm02-ca.crt
[PASS] base URL ends in /v1 (append /chat/completions once)


## 2. List models

In [3]:
models = client.models()
check("GET /models returns Qwen/Qwen3-8B", E["LLM_MODEL"] in models, str(models))

[PASS] GET /models returns Qwen/Qwen3-8B — ['Qwen/Qwen3-8B']


## 3. Basic chat completion

In [4]:
t = time.time()
r = client.chat("Explain RAG in three short steps.", max_tokens=200)
dt = time.time() - t
msg = r["choices"][0]["message"]
print(msg["content"])
print("\nusage:", r["usage"], "| finish_reason:", r["choices"][0]["finish_reason"], f"| {dt:.1f}s")
check("chat returns text", bool(msg["content"]), f"{dt:.1f}s, {r['usage']['completion_tokens']} output tokens")
check("no <think> block when thinking is off", "<think>" not in msg["content"])

1. **Collect and Store Data**: Gather relevant documents or data sources and store them in a structured format for easy access.  
2. **Index and Search**: Use a search engine or indexing system to quickly find the most relevant information when a query is made.  
3. **Generate Response**: Combine the retrieved information with a language model to produce a coherent and accurate answer.

usage: {'prompt_tokens': 21, 'total_tokens': 98, 'completion_tokens': 77, 'prompt_tokens_details': None} | finish_reason: stop | 1.8s
[PASS] chat returns text — 1.8s, 77 output tokens
[PASS] no <think> block when thinking is off


## 4. Thinking mode (Qwen3)

Qwen3 can write out reasoning before it answers. The reasoning counts toward the 1024-token output limit, so the client turns it off by default. This test checks what the server does in each mode.

In [5]:
q = "A train leaves at 14:10 and arrives at 16:55. How long is the trip? Answer in one line."
off = client.chat(q, max_tokens=300, thinking=False)
on  = client.chat(q, max_tokens=1024, thinking=True)
for label, r in [("thinking OFF", off), ("thinking ON", on)]:
    m = r["choices"][0]["message"]
    reasoning = m.get("reasoning_content") or m.get("reasoning") or ""
    print(f"--- {label}: {r['usage']['completion_tokens']} output tokens, finish={r['choices'][0]['finish_reason']}")
    print("reasoning field:", (reasoning[:200] + "…") if reasoning else "(none)")
    print("content:", (m["content"] or "")[:300])
on_m = on["choices"][0]["message"]
think_visible = bool(on_m.get("reasoning_content") or on_m.get("reasoning")) or "<think>" in (on_m["content"] or "")
off_m = off["choices"][0]["message"]
check("thinking OFF: no reasoning in the answer", "<think>" not in (off_m["content"] or "") and not off_m.get("reasoning_content"))
check("thinking ON: reasoning returned", True if think_visible else None,
      f"{on['usage']['completion_tokens']} vs {off['usage']['completion_tokens']} tokens" if think_visible
      else "server returns no reasoning even when asked — thinking appears disabled server-side (saves output tokens)")

--- thinking OFF: 14 output tokens, finish=stop
reasoning field: (none)
content: The trip is 2 hours and 45 minutes long.
--- thinking ON: 14 output tokens, finish=stop
reasoning field: (none)
content: The trip is 2 hours and 45 minutes long.
[PASS] thinking OFF: no reasoning in the answer
[WARN] thinking ON: reasoning returned — server returns no reasoning even when asked — thinking appears disabled server-side (saves output tokens)


## 5. Streaming

In [6]:
t = time.time(); first = None; pieces = []
for piece in client.stream_chat("List three uses of vector databases, one line each.", max_tokens=150):
    if first is None: first = time.time() - t
    pieces.append(piece); print(piece, end="", flush=True)
total = time.time() - t
print(f"\n\nfirst text after {first:.2f}s, done after {total:.2f}s, {len(pieces)} pieces")
check("streaming works", len(pieces) > 1, f"time to first text {first:.2f}s")

1

.

 St

oring

 and

 querying

... (29 outputs left out)

 fraud

 detection

.



first text after 0.13s, done after 1.00s, 40 pieces
[PASS] streaming works — time to first text 0.13s


## 6. Conversation history (multi-turn)

In [7]:
history = [{"role": "system", "content": "You are a concise assistant."},
           {"role": "user", "content": "My group number is 023. Remember it."}]
history.append({"role": "assistant", "content": client.ask(history, max_tokens=60)})
history.append({"role": "user", "content": "What is my group number? Reply with just the number."})
ans = client.ask(history, max_tokens=20)
print(ans)
check("model uses earlier turns", "023" in ans or "23" in ans, ans.strip())

023
[PASS] model uses earlier turns — 023


## 7. Tool calling

In [8]:
tools = [{"type": "function", "function": {
    "name": "search_docs", "description": "Search the course documents and return matching passages.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]
msgs = [{"role": "user", "content": "Use the search tool to find the late submission policy."}]
try:
    r = client.chat(msgs, max_tokens=300, tools=tools, tool_choice="auto")
    m = r["choices"][0]["message"]
    calls = m.get("tool_calls") or []
    print("tool_calls:", json.dumps(calls, indent=1)[:500] if calls else "(none)", "\ncontent:", (m.get("content") or "")[:200])
    check("model returns a tool call", bool(calls))
    if calls:
        msgs += [m, {"role": "tool", "tool_call_id": calls[0]["id"],
                     "content": "Late submissions lose 10% per day, up to 3 days."}]
        final = client.ask(msgs, max_tokens=150, tools=tools)
        print("final answer:", final)
        check("model uses the tool result", "10" in final, final.strip()[:80])
except ApiError as e:
    check("model returns a tool call", False, str(e))

tool_calls: [
 {
  "id": "chatcmpl-tool-b0861ca927348625",
  "type": "function",
  "function": {
   "name": "search_docs",
   "arguments": "{\"query\": \"late submission policy\"}"
  }
 }
] 
content: 
[PASS] model returns a tool call


final answer: The late submission policy states that late submissions lose 10% per day, up to 3 days.
[PASS] model uses the tool result — The late submission policy states that late submissions lose 10% per day, up to 


## 8. Embeddings

In [9]:
v = client.embed(["Example document chunk"])
check("embedding has 384 dimensions", len(v[0]) == 384, f"{len(v[0])} dims")

texts = [f"Sentence number {i} about retrieval." for i in range(40)]
t = time.time(); vs = client.embed(texts)
check("client splits 40 texts into batches of 16", len(vs) == 40, f"{len(vs)} vectors in {time.time()-t:.1f}s")

def cos(a, b): return sum(x*y for x, y in zip(a, b)) / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))
q, near, far = client.embed(["How do I reset my password?",
                             "Steps to change your account password",
                             "The train timetable for Kharagpur station"])
print(f"similar pair {cos(q, near):.3f} | unrelated pair {cos(q, far):.3f}")
check("similar texts score higher than unrelated", cos(q, near) > cos(q, far))

s, body = raw(*EMB, "/embeddings", {"model": E["EMBED_MODEL"], "input": [f"t{i}" for i in range(17)]})
check("server rejects 17 texts in one request", s != 200, f"HTTP {s}: {body[:120]}" if s != 200 else "accepted 17 — limit not enforced")
s, body = raw(*EMB, "/embeddings", {"model": E["EMBED_MODEL"], "input": ["word " * 1000]})
check("server rejects a text over 512 tokens (so chunks must stay under 512)", s != 200, f"HTTP {s}: {body[:120]}")

[PASS] embedding has 384 dimensions — 384 dims


[PASS] client splits 40 texts into batches of 16 — 40 vectors in 0.5s
similar pair 0.835 | unrelated pair 0.388
[PASS] similar texts score higher than unrelated


[PASS] server rejects 17 texts in one request — HTTP 422: {"message":"batch size 17 > maximum allowed batch size 16","code":422,"type":"Validation"}
[PASS] server rejects a text over 512 tokens (so chunks must stay under 512) — HTTP 422: {"message":"Input validation error: `inputs` must have less than 512 tokens. Given: 1002","code":422,"type":"Validation"


## 9. Error cases

Each case is sent with no retry, so you see the server's raw response. The expected codes come from the Track B guide.

In [10]:
chat = lambda **kw: {"model": E["LLM_MODEL"], "messages": [{"role": "user", "content": "hi"}], "max_tokens": 5, **kw}
cases = [
    ("wrong chat key → 401",            401, raw(LLM[0], "wrong-key", LLM[2], "/chat/completions", chat())),
    ("wrong embeddings key → 401",      401, raw(EMB[0], "wrong-key", EMB[2], "/embeddings", {"model": E["EMBED_MODEL"], "input": ["x"]})),
    ("/v1 repeated in path → 404",      404, raw(LLM[0] + "/v1", LLM[1], LLM[2], "/chat/completions", chat())),
    ("wrong model ID → 400/404",        (400, 404), raw(*LLM, "/chat/completions", chat(model="Qwen/Qwen3-70B"))),
    ("max_tokens 1025 → 400",           400, raw(*LLM, "/chat/completions", chat(max_tokens=1025))),
    ("max_tokens 0 → 400",              400, raw(*LLM, "/chat/completions", chat(max_tokens=0))),
    ("input over 8192 tokens → 400",    400, raw(*LLM, "/chat/completions",
                                            {"model": E["LLM_MODEL"], "max_tokens": 10,
                                             "messages": [{"role": "user", "content": "word " * 9000}]})),
    ("malformed JSON → 400/422",        (400, 422), None),
    ("no certificate → SSL error",      "SSL", raw(LLM[0], LLM[1], None, "/models")),
    ("wrong certificate (VM02) → SSL error", "SSL", raw(LLM[0], LLM[1], EMB[2], "/models")),
]
# malformed JSON needs a hand-built request
req = urllib.request.Request(LLM[0] + "/chat/completions", data=b"{not json", method="POST",
                             headers={"Authorization": f"Bearer {LLM[1]}", "Content-Type": "application/json"})
try:
    urllib.request.urlopen(req, context=ssl.create_default_context(cafile=LLM[2]), timeout=30); res = (200, "")
except urllib.error.HTTPError as e: res = (e.code, e.read().decode(errors="replace"))
cases[7] = (cases[7][0], cases[7][1], res)

for name, want, (status, body) in cases:
    if want == "SSL":
        ok = "SSL" in str(status) or "CERTIFICATE" in body.upper()
    else:
        ok = status in (want if isinstance(want, tuple) else (want,))
    check(name, ok, f"got {status}: {body[:100].strip()}")

try:
    client.chat("hi", max_tokens=2000); check("client blocks max_tokens > 1024 before sending", False)
except ValueError as e:
    check("client blocks max_tokens > 1024 before sending", True, str(e))

[PASS] wrong chat key → 401 — got 401: {"detail":"invalid or revoked API key"}
[PASS] wrong embeddings key → 401 — got 401: A valid key for this endpoint is required.
[PASS] /v1 repeated in path → 404 — got 404: {"detail":"endpoint is not available in this pilot"}
[PASS] wrong model ID → 400/404 — got 400: {"detail":"model is not available on this endpoint"}
[PASS] max_tokens 1025 → 400 — got 400: {"detail":"output token limit must be between 1 and 1024"}
[PASS] max_tokens 0 → 400 — got 400: {"detail":"output token limit must be between 1 and 1024"}
[PASS] input over 8192 tokens → 400 — got 400: {"error":{"message":"This model's maximum context length is 8192 tokens. However, you requested 10 o
[PASS] malformed JSON → 400/422 — got 400: {"detail":"request must contain valid JSON"}
[PASS] no certificate → SSL error — got URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issue
[PASS] wrong certificate (VM02) → SSL error — got URLErro

## 10. One active request per key

**10a.** Two raw requests sent at the same moment, with no lock. The guide says the second one gets a 429 error.
**10b.** The same two requests through `client`. Its lock sends them one after the other, so both should succeed.

In [11]:
body = {"model": E["LLM_MODEL"], "max_tokens": 300, "messages": [{"role": "user", "content": "Write a 200-word story about a robot."}]}
out = [None, None]
def go(i): out[i] = raw(*LLM, "/chat/completions", body)[0]
ts = [threading.Thread(target=go, args=(i,)) for i in range(2)]
[t.start() for t in ts]; [t.join() for t in ts]
print("raw parallel statuses:", out)
check("10a: second parallel request gets 429", 429 in out, str(out) if 429 in out else f"{out} — server allowed both")

out2 = [None, None]; t0 = time.time()
def go2(i):
    try: client.chat(body["messages"], max_tokens=300); out2[i] = 200
    except ApiError as e: out2[i] = e.status
ts = [threading.Thread(target=go2, args=(i,)) for i in range(2)]
[t.start() for t in ts]; [t.join() for t in ts]
print("client parallel statuses:", out2, f"in {time.time()-t0:.1f}s")
check("10b: client sends parallel calls one at a time, both succeed", out2 == [200, 200], str(out2))

raw parallel statuses: [200, 429]
[PASS] 10a: second parallel request gets 429 — [200, 429]


client parallel statuses: [200, 200] in 11.9s
[PASS] 10b: client sends parallel calls one at a time, both succeed — [200, 200]


## 11. Retry and rate-limit logic (offline)

Uses a fake server, so it costs no quota. It checks that 429/503 errors are retried after 2, 4 and 8 seconds, that the client stops after 3 retries, and that 401/400 errors are never retried.

In [12]:
import io
class FakeEndpoint(Endpoint):
    def __init__(self, script):
        self.base_url, self.api_key, self.name = "https://fake/v1", "k", "fake"
        self.retry_delays, self.lock, self.slept, self.script = (2, 4, 8), None, [], list(script)
        self.timeout = 1
        self.sleep = self.slept.append
        self.limiter = RateLimiter(sleep=lambda s: None)
    def _open(self, route, body=None, stream=False):
        code = self.script.pop(0)
        if code != 200:
            raise urllib.error.HTTPError(self.base_url + route, code, "err", {}, io.BytesIO(b'{"error":"x"}'))
        return io.BytesIO(b'{"ok": true}')

f = FakeEndpoint([429, 503, 200]);           r = f.request("/x")
check("retries 429 then 503, then succeeds", r == {"ok": True} and f.slept == [2, 4], f"slept {f.slept}")
f = FakeEndpoint([429, 429, 429, 429])
try: f.request("/x"); check("gives up after 3 retries", False)
except ApiError as e: check("gives up after 3 retries", e.status == 429 and f.slept == [2, 4, 8], f"slept {f.slept}")
for code in (401, 400):
    f = FakeEndpoint([code])
    try: f.request("/x")
    except ApiError: pass
    check(f"{code} is not retried", f.slept == [])

clock = [0.0]; waits = []
rl = RateLimiter(limit=60, window=60, clock=lambda: clock[0], sleep=lambda s: (waits.append(s), clock.__setitem__(0, clock[0] + s)))
for i in range(61): rl.wait(); clock[0] += 0.1
check("rate limiter pauses the 61st call within a minute", len(waits) == 1 and 53 < waits[0] < 61, f"waited {waits}")

[PASS] retries 429 then 503, then succeeds — slept [2, 4]
[PASS] gives up after 3 retries — slept [2, 4, 8]
[PASS] 401 is not retried
[PASS] 400 is not retried
[PASS] rate limiter pauses the 61st call within a minute — waited [54.00000000000001]


## 12. Key hygiene

In [13]:
keys = {"LLM_API_KEY": E["LLM_API_KEY"], "EMBED_API_KEY": E["EMBED_API_KEY"]}
ignored = subprocess.run(["git", "check-ignore", ".env", "secrets/vm06-ca.crt"], capture_output=True, text=True).stdout.split()
check(".env and secrets/ are gitignored", ".env" in ignored and "secrets/vm06-ca.crt" in ignored, str(ignored))
for name, k in keys.items():
    hist = subprocess.run(["git", "log", "--all", "-p", "-S", k], capture_output=True, text=True).stdout
    tracked = subprocess.run(["git", "grep", "-l", "-F", k], capture_output=True, text=True).stdout
    check(f"{name} not in git history or tracked files", not hist and not tracked)
nb_text = open("api_tests.ipynb").read() if os.path.exists("api_tests.ipynb") else ""
check("keys not written into this notebook file", not any(k in nb_text for k in keys.values()))

[PASS] .env and secrets/ are gitignored — ['.env', 'secrets/vm06-ca.crt']
[PASS] LLM_API_KEY not in git history or tracked files
[PASS] EMBED_API_KEY not in git history or tracked files
[PASS] keys not written into this notebook file


## 13. Summary

In [14]:
from collections import Counter
print(Counter(s for _, s, _ in RESULTS))
print()
for name, status, detail in RESULTS:
    if status != "PASS": print(f"{status}: {name} — {detail}")

llm_ok = [c for c in CALL_LOG if c["service"] == "llm" and c["status"] == 200 and c.get("usage")]
if llm_ok:
    secs = sorted(c["seconds"] for c in llm_ok)
    out_toks = [c["usage"]["completion_tokens"] for c in llm_ok]
    tps = [c["usage"]["completion_tokens"] / c["seconds"] for c in llm_ok if c["seconds"] > 0.5]
    print(f"\nchat calls via client: {len(llm_ok)} | median {secs[len(secs)//2]:.1f}s | max {secs[-1]:.1f}s"
          + (f" | ~{sum(tps)/len(tps):.0f} output tokens/s" if tps else ""))
print("real API attempts by status:", Counter(c["status"] for c in CALL_LOG if c["service"] != "fake"))

Counter({'PASS': 46, 'WARN': 1})

WARN: thinking ON: reasoning returned — server returns no reasoning even when asked — thinking appears disabled server-side (saves output tokens)

chat calls via client: 9 | median 0.6s | max 6.1s | ~38 output tokens/s
real API attempts by status: Counter({200: 15})
